# Yolo

**YOLO(You Only Look Once)**는 객체 탐지(Object Detection)를 수행하는 실시간 딥러닝 모델로, 이미지나 비디오에서 객체의 **위치**와 **종류**를 동시에 예측할 수 있다. YOLO는 객체 탐지를 단일 신경망 패스로 수행하며, 기존의 슬라이딩 윈도우 방식이나 영역 제안(region proposal) 기반 모델에 비해 빠르고 효율적이다.

- **단일 패스 처리**: 이미지를 한 번만 보고 모든 객체를 탐지한다
- **실시간 처리**: 빠른 속도로 영상 처리가 가능하다
- **통합 네트워크**: 분류와 위치 추정을 동시에 수행한다

**작동 원리:**

**1단계: 이미지 분할**
- 입력 이미지를 그리드(grid)로 나눈다
- 예: 7x7 또는 13x13 그리드로 분할한다

**2단계: 경계 상자 예측**
- 각 그리드 셀이 여러 개의 경계 상자(bounding box)를 예측한다
- 각 상자는 (x, y, width, height, confidence) 정보를 포함한다

**3단계: 클래스 확률 계산**
- 각 그리드 셀이 객체 클래스 확률을 계산한다
- 예: 사람(0.9), 자동차(0.1), 개(0.05) 등

**4단계: 최종 결과 생성**
- 신뢰도가 높은 예측만 선택한다
- 중복된 예측은 NMS(Non-Maximum Suppression)로 제거한다


**YOLO의 주요 활용 분야:**

1. **자율주행**  
   - 차량 주변의 보행자, 차량, 도로 표지판 등을 실시간으로 감지.

2. **보안 및 감시**  
   - CCTV 영상에서 침입자 감지, 이상 행동 탐지.

3. **의료 영상 분석**  
   - X-ray나 MRI 이미지에서 병변 탐지.

4. **리테일 분석**  
   - 매장에서 고객 행동 분석, 상품 식별.

**YOLO와 다른 객체 탐지 모델 비교**

| **특징**               | **YOLO**                          | **Faster R-CNN**               | **SSD(Single Shot Detector)**   |
|------------------------|------------------------------------|---------------------------------|----------------------------------|
| **처리 속도**          | 매우 빠름 (실시간 가능)            | 느림                            | 중간                             |
| **정확도**             | 높음 (특히 최신 버전)             | 매우 높음                       | 높음                             |
| **구조**               | 단일 단계 (End-to-End)            | 두 단계 (Region Proposal + Detection) | 단일 단계                        |
| **복잡도**             | 낮음                              | 높음                            | 낮음                             |

## Yolo v11 객체탐지

https://docs.ultralytics.com/tasks/detect/


![](https://ultralytics.com/images/bus.jpg)

In [2]:
!pip install -Uq ipywidgets ultralytics


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from ultralytics import YOLO

model = YOLO('yolo11n')
model

WARNING Ultralytics settings updated to the latest schema. Existing values were preserved where possible. 
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\playdata2\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=

In [4]:
results = model('https://ultralytics.com/images/bus.jpg')
results[0].show()
results[0].save(filename='results/bus_yolo.jpg')


image 1/1 c:\Users\playdata2\Multimodal\01_cnn\bus.jpg: 640x480 4 persons, 1 bus, 95.4ms
Speed: 4.0ms preprocess, 95.4ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 480)


'results/bus_yolo.jpg'

- cls: 각 박스의 클래스 id(정수 라벨). 예: 0=person, 5=bus(보통 COCO 기준)
- conf: 각 박스의 신뢰도 점수(0~1). 높을수록 더 확신.
- data: 박스 정보를 한 번에 담은 텐서. 보통 [x1, y1, x2, y2, conf, cls] 형태로 행이 박스 1개.
- id: **트래킹(track)**을 쓸 때 객체 id가 들어감. 지금은 None(트래킹 안 함).
- is_track: 현재 결과가 트래킹 결과인지 여부. False면 단순 탐지.
- orig_shape: 원본 이미지 크기 (H, W).
- shape: data 텐서의 크기. 예: [5, 6]이면 박스 5개, 각 박스당 값 6개.
- xyxy: 박스 좌표를 (x1, y1, x2, y2)(좌상단/우하단) 픽셀 기준으로 제공.
- xywh: 박스 좌표를 (x_center, y_center, width, height) 픽셀 기준으로 제공.
- xyxyn: xyxy를 정규화(0~1) 한 값(이미지 너비/높이로 나눈 좌표).
- xywhn: xywh를 정규화(0~1) 한 값.

In [5]:
results

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: None
 names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 

In [6]:
results = model('https://upload.wikimedia.org/wikipedia/commons/7/75/Makati_intersection.jpg')
results[0].show()
results[0].save(filename='results/intersection_yolo.jpg')


WARNING Download failure, retrying 1/3 https://upload.wikimedia.org/wikipedia/commons/7/75/Makati_intersection.jpg... 403 Client Error: Forbidden for url: https://upload.wikimedia.org/wikipedia/commons/7/75/Makati_intersection.jpg
image 1/1 c:\Users\playdata2\Multimodal\01_cnn\Makati_intersection.jpg: 480x640 32 cars, 1 truck, 86.3ms
Speed: 3.3ms preprocess, 86.3ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)


'results/intersection_yolo.jpg'

In [7]:
!pip install gdown

  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached soupsieve-2.9.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)
Using cached tqdm-4.70.0-py3-none-any.whl (80 kB)
Using cached PySocks-1.7.1-py3-none-any.whl (16 kB)
Using cached soupsieve-2.9.2-py3-none-any.whl (37 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# https://drive.google.com/file/d/1tMZODkkNA-ypSuxmGTZgthTG8b3JgFRD
!gdown 1tMZODkkNA-ypSuxmGTZgthTG8b3JgFRD

Using cookies from C:\Users\playdata2\.cache/gdown/cookies.txt
Downloading...
From: https://drive.google.com/uc?id=1tMZODkkNA-ypSuxmGTZgthTG8b3JgFRD
To: c:\Users\playdata2\Multimodal\01_cnn\Night_Day_Chase.mp4

  0%|          | 0.00/23.4M [00:00<?, ?B/s]
  2%|▏         | 524k/23.4M [00:00<00:12, 1.88MB/s]
  7%|▋         | 1.57M/23.4M [00:00<00:05, 4.14MB/s]
 20%|██        | 4.72M/23.4M [00:00<00:01, 12.1MB/s]
 29%|██▉       | 6.82M/23.4M [00:00<00:01, 14.5MB/s]
 47%|████▋     | 11.0M/23.4M [00:00<00:00, 14.2MB/s]
 65%|██████▌   | 15.2M/23.4M [00:01<00:00, 19.6MB/s]
 83%|████████▎ | 19.4M/23.4M [00:01<00:00, 23.2MB/s]
100%|██████████| 23.4M/23.4M [00:01<00:00, 18.6MB/s]


In [10]:
video_path = 'Night_Day_Chase.mp4'
results = model(video_path, save=True)


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/1383) c:\Users\playdata2\Multimodal\01_cnn\Night_Day_Chase.mp4: 288x640 2 persons, 1 motorcycle, 76.0ms
video 1/1 (frame 2/1383) c:\Users\playdata2\Multimodal\01_cnn\Night_Day_Chase.mp4: 288x640 2 persons, 1 motorcycle, 55.3ms
video 1/1 (frame 3/1383) c:\Users\playdata2\Multimodal\01_cnn\Night_Day_Chase.mp4: 288x640 2 persons, 1 motorcycle, 31.2ms
video 1/1 (frame 4/1383) c:\Users\playdata2\Multimodal\01_cnn\Night_Day_Chase.mp4: 288x640 2 

In [11]:
import cv2
import numpy as np
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

def get_colors(num_colors):
    np.random.seed(0)
    return [tuple(np.random.randint(0, 255, 3).tolist()) for _ in  range(num_colors)]

class_names = model.names
num_classes = len(class_names)
colors = get_colors(num_classes)
print(class_names)

{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'cell phone', 68: 'microw

In [ ]:
def detect_objects(image: np.array):
    